**Azure Databricks — Hands-on Assessment**

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (session_id INT, zone_id STRING, vehicle_type STRING, duration_hours INT, parking_fee INT, status STRING) USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101,'Z01','Car',3,150,'Completed'),(102,'Z02','Bike',2,40,'Completed'),(103,'Z03','Car',5,250,'Completed'),(104,'Z01','SUV',4,240,'Active'),(105,'Z04','Bike',1,20,'Completed'),
(106,'Z02','Car',6,300,'Completed'),(107,'Z03','SUV',2,120,'Cancelled'),(108,'Z05','Car',3,150,'Completed'),(109,'Z01','Bike',4,80,'Completed'),(110,'Z04','SUV',5,300,'Active'),
(111,'Z02','Car',2,100,'Completed'),(112,'Z05','Bike',3,60,'Cancelled'),(113,'Z03','Car',7,350,'Completed'),(114,'Z01','SUV',3,180,'Completed'),(115,'Z04','Bike',2,40,'Active'),
(116,'Z05','Car',4,200,'Completed'),(117,'Z02','SUV',6,360,'Completed'),(118,'Z03','Bike',1,20,'Completed'),(119,'Z04','Car',5,250,'Completed'),(120,'Z05','SUV',2,120,'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work USING DELTA AS SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (zone_id STRING, zone_name STRING, city STRING, capacity INT) USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES ('Z01','Central Plaza','Hyderabad',100),('Z02','Metro Parking','Bengaluru',150),('Z03','City Mall','Mumbai',200),('Z04','Airport Parking','Delhi',250),('Z05','Tech Park','Pune',120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS SELECT * FROM VALUES
(102,'Z02','Bike',3,60,'Completed'),(104,'Z01','SUV',5,300,'Completed'),(121,'Z03','Car',4,200,'Completed'),(122,'Z05','Bike',2,40,'Active')
AS t(session_id,zone_id,vehicle_type,duration_hours,parking_fee,status);

In [0]:
%sql
SELECT * FROM parking_sessions_source; 
SELECT session_id, vehicle_type, parking_fee FROM parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
SELECT * FROM parking_sessions_source WHERE status='Completed' AND parking_fee>100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
SELECT vehicle_type, SUM(parking_fee) AS total_parking_fees, COUNT(*) AS completed_sessions FROM parking_sessions_source WHERE status='Completed' GROUP BY vehicle_type; 

vehicle_type,total_parking_fees,completed_sessions
Car,1750,8
Bike,160,4
SUV,540,2


In [0]:
from pyspark.sql import functions as F
parking_df = spark.table("parking_sessions_source")
parking_category_df = parking_df.withColumn("fee_category", F.when(F.col("parking_fee") >= 200, "High").otherwise("Low"))
display(parking_category_df)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


In [0]:
%sql
CREATE OR REPLACE TABLE parking_summary USING DELTA AS SELECT * FROM parking_sessions_source WHERE status='Completed'; 

num_affected_rows,num_inserted_rows


In [0]:
%sql
INSERT INTO parking_sessions_work VALUES (123,'Z01','Car',3,150,'Active'); 

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE parking_sessions_work SET status='Completed', parking_fee=280 WHERE session_id=104; 

num_affected_rows
1


In [0]:
%sql
DELETE FROM parking_sessions_work WHERE status='Cancelled'; 
SELECT * FROM parking_sessions_work ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,280,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


In [0]:
%sql
SELECT * FROM incoming_parking_sessions; 
SELECT i.* FROM incoming_parking_sessions i INNER JOIN parking_sessions_work w ON i.session_id=w.session_id; 

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-06T04:48:59.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(229690149719177),aec0c846-e2f5-4b1f-ad6e-ebdd2d0ec89c,1106-034438-t2etd6jv-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6092, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T04:48:57.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#23670 = Cancelled)""])",null,List(229690149719177),aec0c846-e2f5-4b1f-ad6e-ebdd2d0ec89c,1106-034438-t2etd6jv-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1020, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 575, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 444)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T04:48:41.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(session_id#23170 = 104)""])",null,List(229690149719177),9e1e7ea7-4dfa-48b9-a89d-a01c92fedc71,1106-034438-t2etd6jv-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3310, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1622, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1932, rewriteTimeMs -> 1662)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T04:48:16.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(229690149719177),de0e67ba-c265-41a7-b992-813c8cb32866,1106-034438-t2etd6jv-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1878)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-06T04:42:52.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(229690149719177),e596d77b-61a3-47c0-af9b-22b83ccb6f95,1106-034438-t2etd6jv-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 2282)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
MERGE INTO parking_sessions_work t USING incoming_parking_sessions s ON t.session_id=s.session_id
WHEN MATCHED THEN UPDATE SET t.zone_id=s.zone_id,t.vehicle_type=s.vehicle_type,t.duration_hours=s.duration_hours,t.parking_fee=s.parking_fee,t.status=s.status
WHEN NOT MATCHED THEN INSERT (session_id,zone_id,vehicle_type,duration_hours,parking_fee,status)
VALUES (s.session_id,s.zone_id,s.vehicle_type,s.duration_hours,s.parking_fee,s.status); 

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T04:53:12.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(229690149719177),2f563289-87b1-4d34-9d75-02927257c9fc,1106-034438-t2etd6jv-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9875, p25FileSize -> 2287, numDeletionVectorsRemoved -> 1, minFileSize -> 2287, numAddedFiles -> 1, maxFileSize -> 2287, p75FileSize -> 2287, p50FileSize -> 2287, numAddedBytes -> 2287)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T04:53:11.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(session_id#24486 = session_id#24468)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(229690149719177),2f563289-87b1-4d34-9d75-02927257c9fc,1106-034438-t2etd6jv-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7597, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3353, materializeSourceTimeMs -> 272, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1154, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1863)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T04:48:59.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(229690149719177),aec0c846-e2f5-4b1f-ad6e-ebdd2d0ec89c,1106-034438-t2etd6jv-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6092, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T04:48:57.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#23670 = Cancelled)""])",null,List(229690149719177),aec0c846-e2f5-4b1f-ad6e-ebdd2d0ec89c,1106-034438-t2etd6jv-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1020, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 575, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 444)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T04:48:41.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(session_id#23170 = 104)""])",null,List(229690149719177),9e1e7ea7-4dfa-48b9-a89d-a01c92fedc71,1106-034438-t2etd6jv-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3310, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1622, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1932, rewriteTimeMs -> 1662)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T04:48:16.000Z,144976867014371,azuser7407_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append,

In [0]:
from pyspark.sql import functions as F

history_df = spark.sql("DESCRIBE HISTORY parking_sessions_work")

merge_record = (
    history_df
    .filter(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc())
    .first())

In [0]:
%sql
SELECT *
FROM parking_sessions_work VERSION AS OF 6
ORDER BY session_id=102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
122,Z05,Bike,2,40,Active
118,Z03,Bike,1,20,Completed
123,Z01,Car,3,150,Active
121,Z03,Car,4,200,Completed
115,Z04,Bike,2,40,Active
117,Z02,SUV,6,360,Completed
114,Z01,SUV,3,180,Completed
120,Z05,SUV,2,120,Active
104,Z01,SUV,5,300,Completed
111,Z02,Car,2,100,Completed


In [0]:
%sql
SELECT * FROM parking_sessions_work WHERE session_id=102; 

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed


In [0]:
%sql
CREATE OR REPLACE VIEW completed_parking_view AS SELECT * FROM parking_sessions_work WHERE status='Completed'; 
SELECT * FROM completed_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW z01_parking_view AS
SELECT *
FROM parking_sessions_work
WHERE zone_id = 'Z01';

SELECT * FROM z01_parking_view;

CREATE OR REPLACE TEMP VIEW high_fee_parking_view AS
SELECT *
FROM parking_sessions_work
WHERE parking_fee > 200;

SELECT * FROM high_fee_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
110,Z04,SUV,5,300,Active
113,Z03,Car,7,350,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
SELECT z.zone_name,z.city,w.vehicle_type,w.parking_fee FROM parking_sessions_work w JOIN parking_zones z ON w.zone_id=z.zone_id; 

zone_name,city,vehicle_type,parking_fee
Central Plaza,Hyderabad,Car,150
City Mall,Mumbai,Car,250
Airport Parking,Delhi,Bike,20
Metro Parking,Bengaluru,Car,300
Tech Park,Pune,Car,150
Central Plaza,Hyderabad,Bike,80
Airport Parking,Delhi,SUV,300
Metro Parking,Bengaluru,Car,100
City Mall,Mumbai,Car,350
Central Plaza,Hyderabad,SUV,180


In [0]:
%sql
DESCRIBE DETAIL parking_sessions_work; 
OPTIMIZE parking_sessions_work;
DESCRIBE DETAIL parking_sessions_work; 

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,098cef14-4139-4cd2-a24f-4c13a376d556,kimoo_red.default.parking_sessions_work,null,,2026-11-06T04:42:50.538Z,2026-11-06T04:53:12.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
details = spark.sql("""
    SELECT current_catalog() AS catalog,
           current_schema() AS schema""").first()

catalog = details["catalog"]
schema = details["schema"]

print("Catalog:", catalog)
print("Schema:", schema)




Catalog: kimoo_red
Schema: default


In [0]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load())

query = (
    rate_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"/Volumes/{catalog}/{schema}/parking_checkpoints/q17_append"
    )
    .trigger(availableNow=True)
    .toTable("parking_rate_output"))

display(spark.table("parking_rate_output"))

timestamp,value


In [0]:
print("Query status:", query.status)
print("Query active:", query.isActive)

display(spark.table("parking_rate_output"))

Query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Query active: False


timestamp,value
2026-11-06T05:22:34.147Z,0
2026-11-06T05:22:34.347Z,1


In [0]:
amount_stream_df = (
    rate_df
    .withColumn("amount", F.col("value").cast("long"))
    .filter(F.col("amount") > 300)
)

print("Is streaming DataFrame:", amount_stream_df.isStreaming)

query_q18 = (
    amount_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"/Volumes/{catalog}/{schema}/parking_checkpoints/q18_append"
    )
    .trigger(availableNow=True)
    .toTable("parking_amount_output")
)

display(spark.table("parking_amount_output"))

Is streaming DataFrame: True


timestamp,value,amount


In [0]:
%sql


CREATE OR REFRESH STREAMING TABLE parking_bronze
AS
SELECT *
FROM STREAM(parking_sessions_source);




CREATE OR REFRESH STREAMING TABLE parking_silver
AS
SELECT *
FROM STREAM(parking_bronze)
WHERE status = 'Completed';

message
The operation was successfully executed.


In [0]:
%sql


CREATE OR REFRESH MATERIALIZED VIEW parking_gold_by_zone
AS
SELECT
    zone_id,
    COUNT(*) AS total_completed_sessions,
    SUM(parking_fee) AS total_parking_revenue
FROM parking_silver
GROUP BY zone_id;

result
The operation was successfully executed.


In [0]:
%sql

SELECT *
FROM kimoo_red.default.parking_gold_by_zone
ORDER BY zone_id;

zone_id,total_completed_sessions,total_parking_revenue
Z01,3,410
Z02,4,800
Z03,3,620
Z04,2,270
Z05,2,350
